# ITC Form Data Analysis

This notebook analyzes the `idc_acc_form_responses_datamart` Hive table to understand:
1. Equipment types in the data
2. Sections/categories used for each equipment type
3. Common check items for each type
4. Acceptance criteria patterns
5. Typical form structure

The insights will be used to improve the ITC Form Generator.

In [ ]:
# Install required packages for Presto/Hive connection
# Uncomment and run if needed:
# !pip install presto-python-client pyhive

import os
from typing import Optional

# Check for available database connection methods
try:
    from pyhive import presto
    print("PyHive available for Presto connection")
except ImportError:
    print("PyHive not available")

try:
    import prestodb
    print("prestodb available")
except ImportError:
    print("prestodb not available")

# For Meta internal, we typically use:
# - fblearner (Bento notebooks)
# - presto-cli command line
# - PyHive library
print("\nWill attempt connection methods available in this environment")

## Query 1: Explore Table Schema

First, let's understand the structure of the `idc_acc_form_responses_datamart` table.

In [ ]:
# SQL Query to describe the table schema
schema_query = """
DESCRIBE idc_acc_form_responses_datamart
"""

print("Query to run:")
print(schema_query)

# If using command-line presto-cli, the command would be:
print("\nCommand-line equivalent:")
print('presto-cli --catalog hive --schema default -e "DESCRIBE idc_acc_form_responses_datamart"')

## Query 2: Equipment Types Distribution

Identify all distinct equipment types and their frequency in the data.

In [ ]:
# SQL Query to get equipment types
equipment_types_query = """
SELECT DISTINCT
    equipment_type,
    COUNT(*) as count
FROM idc_acc_form_responses_datamart
GROUP BY equipment_type
ORDER BY count DESC
LIMIT 50
"""

print("Query to run:")
print(equipment_types_query)

## Query 3: Sections/Categories by Equipment Type

Analyze what sections or categories are used for each equipment type.

In [ ]:
# SQL Query to analyze sections/categories
sections_query = """
SELECT
    equipment_type,
    section_name,
    category,
    COUNT(*) as item_count
FROM idc_acc_form_responses_datamart
WHERE equipment_type IS NOT NULL
GROUP BY equipment_type, section_name, category
ORDER BY equipment_type, item_count DESC
"""

print("Query to run:")
print(sections_query)

## Query 4: Common Check Items by Equipment Type

Identify the most common check items for each equipment type.

In [ ]:
# SQL Query to find common check items
check_items_query = """
SELECT
    equipment_type,
    check_item,
    COUNT(*) as frequency
FROM idc_acc_form_responses_datamart
WHERE equipment_type IS NOT NULL
    AND check_item IS NOT NULL
GROUP BY equipment_type, check_item
ORDER BY equipment_type, frequency DESC
"""

print("Query to run:")
print(check_items_query)

# Query for top check items per equipment type (more focused)
top_items_query = """
WITH ranked_items AS (
    SELECT
        equipment_type,
        check_item,
        COUNT(*) as frequency,
        ROW_NUMBER() OVER (PARTITION BY equipment_type ORDER BY COUNT(*) DESC) as rn
    FROM idc_acc_form_responses_datamart
    WHERE equipment_type IS NOT NULL
        AND check_item IS NOT NULL
    GROUP BY equipment_type, check_item
)
SELECT
    equipment_type,
    check_item,
    frequency
FROM ranked_items
WHERE rn <= 10
ORDER BY equipment_type, frequency DESC
"""

print("\nTop 10 check items per equipment type:")
print(top_items_query)

## Query 5: Acceptance Criteria Patterns

Analyze the acceptance criteria patterns used across different equipment types.

In [ ]:
# SQL Query to analyze acceptance criteria patterns
acceptance_criteria_query = """
SELECT
    equipment_type,
    acceptance_criteria,
    COUNT(*) as usage_count
FROM idc_acc_form_responses_datamart
WHERE acceptance_criteria IS NOT NULL
GROUP BY equipment_type, acceptance_criteria
ORDER BY equipment_type, usage_count DESC
"""

print("Query to run:")
print(acceptance_criteria_query)

# Alternative: Look for patterns in acceptance criteria
criteria_patterns_query = """
SELECT
    CASE
        WHEN LOWER(acceptance_criteria) LIKE '%pass%' THEN 'Pass/Fail'
        WHEN LOWER(acceptance_criteria) LIKE '%verify%' THEN 'Verification'
        WHEN LOWER(acceptance_criteria) LIKE '%range%' OR acceptance_criteria LIKE '%-%' THEN 'Range Check'
        WHEN LOWER(acceptance_criteria) LIKE '%visual%' THEN 'Visual Inspection'
        WHEN LOWER(acceptance_criteria) LIKE '%measure%' THEN 'Measurement'
        WHEN LOWER(acceptance_criteria) LIKE '%confirm%' THEN 'Confirmation'
        ELSE 'Other'
    END as criteria_type,
    COUNT(*) as count
FROM idc_acc_form_responses_datamart
WHERE acceptance_criteria IS NOT NULL
GROUP BY
    CASE
        WHEN LOWER(acceptance_criteria) LIKE '%pass%' THEN 'Pass/Fail'
        WHEN LOWER(acceptance_criteria) LIKE '%verify%' THEN 'Verification'
        WHEN LOWER(acceptance_criteria) LIKE '%range%' OR acceptance_criteria LIKE '%-%' THEN 'Range Check'
        WHEN LOWER(acceptance_criteria) LIKE '%visual%' THEN 'Visual Inspection'
        WHEN LOWER(acceptance_criteria) LIKE '%measure%' THEN 'Measurement'
        WHEN LOWER(acceptance_criteria) LIKE '%confirm%' THEN 'Confirmation'
        ELSE 'Other'
    END
ORDER BY count DESC
"""

print("\nAcceptance criteria patterns:")
print(criteria_patterns_query)

## Query 6: Form Structure Analysis

Understand the typical form structure including number of sections and items.

In [ ]:
# SQL Query to analyze form structure
form_structure_query = """
SELECT
    equipment_type,
    COUNT(DISTINCT form_id) as num_forms,
    COUNT(DISTINCT section_name) as num_sections,
    COUNT(DISTINCT check_item) as num_check_items,
    AVG(CAST(item_count AS DOUBLE)) as avg_items_per_form
FROM (
    SELECT
        equipment_type,
        form_id,
        section_name,
        check_item,
        COUNT(*) OVER (PARTITION BY form_id) as item_count
    FROM idc_acc_form_responses_datamart
    WHERE equipment_type IS NOT NULL
)
GROUP BY equipment_type
ORDER BY num_forms DESC
"""

print("Query to run:")
print(form_structure_query)

## Query 7: Sample Forms for Each Equipment Type

Get sample data for representative equipment types.

In [ ]:
# SQL Query to get sample form data
sample_mua_query = """
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%MUA%'
   OR UPPER(equipment_type) LIKE '%MAKE%UP%AIR%'
LIMIT 100
"""

sample_ahu_query = """
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%AHU%'
   OR UPPER(equipment_type) LIKE '%AIR%HANDL%'
LIMIT 100
"""

sample_fcu_query = """
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%FCU%'
   OR UPPER(equipment_type) LIKE '%FAN%COIL%'
LIMIT 100
"""

sample_chiller_query = """
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%CHILL%'
LIMIT 100
"""

print("Sample queries for specific equipment types:")
print("\n=== MUA (Makeup Air Unit) ===")
print(sample_mua_query)
print("\n=== AHU (Air Handling Unit) ===")
print(sample_ahu_query)
print("\n=== FCU (Fan Coil Unit) ===")
print(sample_fcu_query)
print("\n=== Chiller ===")
print(sample_chiller_query)

## Query 8: Complete Column Analysis

Get comprehensive understanding of all columns and their possible values.

In [ ]:
# SQL queries to understand all column values

# Priority/severity levels if they exist
priority_query = """
SELECT DISTINCT priority_level, COUNT(*) as count
FROM idc_acc_form_responses_datamart
WHERE priority_level IS NOT NULL
GROUP BY priority_level
ORDER BY count DESC
"""

# Response types if they exist
response_types_query = """
SELECT DISTINCT response_type, COUNT(*) as count
FROM idc_acc_form_responses_datamart
WHERE response_type IS NOT NULL
GROUP BY response_type
ORDER BY count DESC
"""

# Site/location analysis
site_query = """
SELECT DISTINCT
    site_id,
    site_name,
    COUNT(DISTINCT form_id) as num_forms
FROM idc_acc_form_responses_datamart
GROUP BY site_id, site_name
ORDER BY num_forms DESC
LIMIT 20
"""

print("Column analysis queries:")
print("\n=== Priority Levels ===")
print(priority_query)
print("\n=== Response Types ===")
print(response_types_query)
print("\n=== Sites ===")
print(site_query)

## Combined All Queries for Easy Copy-Paste

Run all queries together for analysis.

In [ ]:
# Combined queries for running in Presto CLI or other SQL tool

all_queries = """
-- ===============================================
-- ITC FORM ANALYSIS QUERIES
-- Table: idc_acc_form_responses_datamart
-- ===============================================

-- 1. TABLE SCHEMA
DESCRIBE idc_acc_form_responses_datamart;

-- 2. EQUIPMENT TYPES DISTRIBUTION
SELECT DISTINCT
    equipment_type,
    COUNT(*) as count
FROM idc_acc_form_responses_datamart
GROUP BY equipment_type
ORDER BY count DESC
LIMIT 50;

-- 3. SECTIONS BY EQUIPMENT TYPE
SELECT
    equipment_type,
    section_name,
    category,
    COUNT(*) as item_count
FROM idc_acc_form_responses_datamart
WHERE equipment_type IS NOT NULL
GROUP BY equipment_type, section_name, category
ORDER BY equipment_type, item_count DESC;

-- 4. TOP CHECK ITEMS PER EQUIPMENT TYPE
WITH ranked_items AS (
    SELECT
        equipment_type,
        check_item,
        COUNT(*) as frequency,
        ROW_NUMBER() OVER (PARTITION BY equipment_type ORDER BY COUNT(*) DESC) as rn
    FROM idc_acc_form_responses_datamart
    WHERE equipment_type IS NOT NULL
        AND check_item IS NOT NULL
    GROUP BY equipment_type, check_item
)
SELECT
    equipment_type,
    check_item,
    frequency
FROM ranked_items
WHERE rn <= 10
ORDER BY equipment_type, frequency DESC;

-- 5. ACCEPTANCE CRITERIA PATTERNS
SELECT
    CASE
        WHEN LOWER(acceptance_criteria) LIKE '%pass%' THEN 'Pass/Fail'
        WHEN LOWER(acceptance_criteria) LIKE '%verify%' THEN 'Verification'
        WHEN LOWER(acceptance_criteria) LIKE '%range%' OR acceptance_criteria LIKE '%-%' THEN 'Range Check'
        WHEN LOWER(acceptance_criteria) LIKE '%visual%' THEN 'Visual Inspection'
        WHEN LOWER(acceptance_criteria) LIKE '%measure%' THEN 'Measurement'
        WHEN LOWER(acceptance_criteria) LIKE '%confirm%' THEN 'Confirmation'
        ELSE 'Other'
    END as criteria_type,
    COUNT(*) as count
FROM idc_acc_form_responses_datamart
WHERE acceptance_criteria IS NOT NULL
GROUP BY 1
ORDER BY count DESC;

-- 6. FORM STRUCTURE OVERVIEW
SELECT
    equipment_type,
    COUNT(DISTINCT form_id) as num_forms,
    COUNT(DISTINCT section_name) as num_sections,
    COUNT(DISTINCT check_item) as num_unique_items
FROM idc_acc_form_responses_datamart
WHERE equipment_type IS NOT NULL
GROUP BY equipment_type
ORDER BY num_forms DESC;

-- 7. SAMPLE MUA FORMS
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%MUA%'
   OR UPPER(equipment_type) LIKE '%MAKE%UP%AIR%'
LIMIT 50;

-- 8. SAMPLE AHU FORMS
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%AHU%'
   OR UPPER(equipment_type) LIKE '%AIR%HANDL%'
LIMIT 50;

-- 9. SAMPLE FCU FORMS
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%FCU%'
   OR UPPER(equipment_type) LIKE '%FAN%COIL%'
LIMIT 50;

-- 10. SAMPLE CHILLER FORMS
SELECT *
FROM idc_acc_form_responses_datamart
WHERE UPPER(equipment_type) LIKE '%CHILL%'
LIMIT 50;

"""

print(all_queries)

# Save queries to a file for easy use
with open('itc_analysis_queries.sql', 'w') as f:
    f.write(all_queries)
print("\n" + "="*50)
print("Queries saved to: itc_analysis_queries.sql")
print("="*50)